In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import datetime as dt
pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/e-commerce-customer-for-behavior-analysis/ecommerce_customer_data_large.csv


In [2]:
df=pd.read_csv("/kaggle/input/e-commerce-customer-for-behavior-analysis/ecommerce_customer_data_large.csv")

In [3]:
df.head()

,Customer ID,Purchase Date,Product Category,Product Price,Quantity,Total Purchase Amount,Payment Method,Customer Age,Returns,Customer Name,Age,Gender,Churn
0,44605,2023-05-03 21:30:02,Home,177,1,2427,PayPal,31,1.000,John Rivera,31,Female,0
1,44605,2021-05-16 13:57:44,Electronics,174,3,2448,PayPal,31,1.000,John Rivera,31,Female,0
2,44605,2020-07-13 06:16:57,Books,413,1,2345,Credit Card,31,1.000,John Rivera,31,Female,0
3,44605,2023-01-17 13:14:36,Electronics,396,3,937,Cash,31,0.000,John Rivera,31,Female,0
4,44605,2021-05-01 11:29:27,Books,259,4,2598,PayPal,31,1.000,John Rivera,31,Female,0


In [4]:
df.isnull().sum()

Customer ID                  0
Purchase Date                0
Product Category             0
Product Price                0
Quantity                     0
Total Purchase Amount        0
Payment Method               0
Customer Age                 0
Returns                  47382
Customer Name                0
Age                          0
Gender                       0
Churn                        0
dtype: int64

In [5]:
df.shape

(250000, 13)

### We don't need the returned products, we can remove them from our dataset.

In [6]:
df = df[df["Returns"] != 1.000]


## RFM Analysis is a technique used for customer segmentation.
### It allows grouping customers based on their purchase behavior, enabling the development of strategies for each group.

#### **Recency** (How recently a customer has made a purchase)

#### **Frequency** (How often a customer makes purchases)

#### **Monetary** (How much money a customer spends)



In [7]:
df["Purchase Date"].max()



'2023-09-13 18:42:49'

In [8]:
df = df.rename(columns={"Purchase Date": "Purchase_Date"})
df = df.rename(columns={"Customer ID": "Customer_ID"})
df = df.rename(columns={"Total Purchase Amount": "Total_Purchase_Amount"})
df['Purchase_Date'] = pd.to_datetime(df['Purchase_Date'])
today_date = dt.datetime(2023, 9, 15)

In [9]:
rfm = df.groupby('Customer_ID').agg({'Purchase_Date': lambda Purchase_Date: (today_date - Purchase_Date.max()).days,
                                     'Customer_ID': lambda Customer_ID: Customer_ID.value_counts(),
                                     'Total_Purchase_Amount': lambda Total_Purchase_Amount: Total_Purchase_Amount.sum()})

In [10]:
rfm.head()

,Purchase_Date,Customer_ID,Total_Purchase_Amount
Customer_ID,,,
1,289,3,6290
2,141,2,5381
3,223,4,9423
4,687,2,1252
5,425,2,2088


In [11]:
rfm.columns = ['Recency', 'Frequency', 'Monetary']

## Let's calculate the RFM scores and segment these scores.

In [12]:
rfm["recency_score"] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1])

# 0-100, 0-20, 20-40, 40-60, 60-80, 80-100

rfm["frequency_score"] = pd.qcut(rfm['Frequency'].rank(method="first"), 5, labels=[1, 2, 3, 4, 5])

rfm["monetary_score"] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5])

rfm["RFM_SCORE"] = (rfm['recency_score'].astype(str) +
                    rfm['frequency_score'].astype(str))

In [13]:
seg_map = {
    r'[1-2][1-2]': 'Hibernating',
    r'[1-2][3-4]': 'At Risk',
    r'[1-2]5': 'Cant Loose',
    r'3[1-2]': 'About to sleep',
    r'33': 'Need Attention',
    r'[3-4][4-5]': 'Loyal Customers',
    r'41': 'Promising',
    r'51': 'New Customers',
    r'[4-5][2-3]': 'Potential Loyalists',
    r'5[4-5]': 'Champions'
}


In [14]:
rfm['segment'] = rfm['RFM_SCORE'].replace(seg_map, regex=True)

In [15]:
rfm.head()

,Recency,Frequency,Monetary,recency_score,frequency_score,monetary_score,RFM_SCORE,segment
Customer_ID,,,,,,,,
1,289,3,6290,3,2,2,32,About to sleep
2,141,2,5381,4,1,2,41,Promising
3,223,4,9423,3,4,4,34,Loyal Customers
4,687,2,1252,1,1,1,11,Hibernating
5,425,2,2088,2,1,1,21,Hibernating
